In [5]:
from google.colab import drive
drive.mount('/content/drive')

# 设置你的数据目录
base_dir = '/content/drive/MyDrive/Colab Notebooks'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install pandas numpy scikit-learn matplotlib torch


In [13]:
import os, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

# Configure matplotlib to support Chinese characters
plt.rcParams['font.sans-serif'] = ['SimHei']  # Use SimHei font
plt.rcParams['axes.unicode_minus'] = False  # Correctly display negative signs

# 股票列表
stock_codes = ['601138','002364','002475','603725','601288']
start_date = "20210924"
end_date = datetime.today().strftime("%Y%m%d") # Ensure end_date is current date
window_size = 60
backtest_days = 10
save_dir = "./models"
chart_dir = "./charts"
os.makedirs(save_dir, exist_ok=True)
os.makedirs(chart_dir, exist_ok=True)

# 标准化列名
def standardize_columns(df):
    column_mapping = {
        "日期": "Date", "股票代码": "Code", "开盘": "Open", "收盘": "Close",
        "最高": "High", "最低": "Low", "成交量": "Volume", "成交额": "Amount",
        "振幅": "Amplitude", "涨跌幅": "ChangePct", "涨跌额": "ChangeAmt", "换手率": "Turnover",
        "opendate": "Date", "netamount": "MainNetInflow", "ratioamount": "MainInflowRatio"
    }
    df.rename(columns=column_mapping, inplace=True)
    return df

# RSI计算
def compute_rsi(series, window=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

# 数据加载与特征构造
def load_and_prepare_data(code, start_date, end_date):
    prefix = code[-6:]
    market_prefix = "sz" if code.startswith("0") else ("sh" if code.startswith("6") else "") # Added handling for '6' prefix
    daily_path = f"/content/drive/MyDrive/Colab Notebooks/kline/{prefix}_daily.csv" # Corrected path
    fundflow_path = f"/content/drive/MyDrive/Colab Notebooks/jiqi_do/jiqi_do/{market_prefix}{code}.csv" # Corrected path

    if not os.path.exists(daily_path):
        print(f"⚠️ 缺失数据文件：{daily_path}")
        return None, None, None, None, None, None, None, None

    df = pd.read_csv(daily_path)
    df = standardize_columns(df) # Call standardize_columns here
    df["Date"] = pd.to_datetime(df["Date"])
    df.sort_values("Date", inplace=True)
    df.set_index("Date", inplace=True)

    if os.path.exists(fundflow_path):
        df_fund = pd.read_csv(fundflow_path)
        df_fund = standardize_columns(df_fund)
        df_fund["Date"] = pd.to_datetime(df_fund["Date"])
        df_fund.set_index("Date", inplace=True)
        df = df.merge(df_fund[["MainNetInflow", "MainInflowRatio"]], left_index=True, right_index=True, how="left")
    else:
        print(f"⚠️ 缺失资金流文件：{fundflow_path}")

    # 统一资金流单位（转换为百万）
    if "MainNetInflow" in df.columns:
        df["MainNetInflow"] = df["MainNetInflow"] / 1e6

    # 技术指标
    df["MA5"] = df["Close"].rolling(window=5).mean()
    df["MA10"] = df["Close"].rolling(window=10).mean()
    df["EMA12"] = df["Close"].ewm(span=12, adjust=False).mean()
    df["EMA26"] = df["Close"].ewm(span=26, adjust=False).mean()
    df["MACD"] = df["EMA12"] - df["EMA26"]
    df["RSI"] = compute_rsi(df["Close"], window=14)
    df["Return_1d"] = df["Close"].pct_change()
    df["Volatility_5d"] = df["Close"].rolling(window=5).std()
    df["Weekday"] = df.index.weekday
    df["IsMonthEnd"] = df.index.is_month_end.astype(int)

    # 资金流衍生特征
    if "MainNetInflow" in df.columns and "MainInflowRatio" in df.columns:
        df["MainNetInflow_Smooth"] = df["MainNetInflow"].ewm(span=5, adjust=False).mean()
        df["MainNetInflow_Normalized"] = (df["MainNetInflow"] - df["MainNetInflow"].mean()) / df["MainNetInflow"].std()
        # Use .fillna(0) or .dropna() as pct_change can produce NaNs
        df["MainNetInflow_Ratio_Change"] = df["MainInflowRatio"].pct_change().fillna(0)
        df["MainNetInflow_5d_Std"] = df["MainNetInflow"].rolling(window=5).std()
    else:
        # Initialize these columns with 0 if main funds flow data is missing
        df["MainNetInflow_Smooth"] = 0
        df["MainNetInflow_Normalized"] = 0
        df["MainNetInflow_Ratio_Change"] = 0
        df["MainNetInflow_5d_Std"] = 0
        print("⚠️ Funds flow columns not found, initializing derivative features to 0.")


    # 异动信号（涨停/跌停/资金突变）
    # 涨停标记：假设涨停是接近最高价且涨幅较大
    df["IsLimitUp"] = ((df["High"] >= df["Close"] * 0.99) & (df["ChangePct"] > 9.5)).astype(int)
    # 跌停标记：假设跌停是接近最低价且跌幅较大
    df["IsLimitDown"] = ((df["Low"] <= df["Close"] * 1.01) & (df["ChangePct"] < -9.5)).astype(int)

    # 连续涨停次数 (Simplified: count consecutive IsLimitUp)
    # This requires a more sophisticated approach for true consecutive counting,
    # but a simple rolling sum can indicate recent frequency.
    df["ConsecutiveLimitUp_5d"] = df["IsLimitUp"].rolling(window=5).sum()

    # 主力资金流入突变：资金净流入 > 均值 + 2σ (requires MainNetInflow)
    if "MainNetInflow" in df.columns:
        mean_inflow = df["MainNetInflow"].mean()
        std_inflow = df["MainNetInflow"].std()
        df["IsAbnormalInflow"] = (df["MainNetInflow"] > mean_inflow + 2 * std_inflow).astype(int)
    else:
        df["IsAbnormalInflow"] = 0
        print("⚠️ MainNetInflow column not found, skipping IsAbnormalInflow feature.")

    # 换手率激增：换手率 > 历史均值 + 3σ
    if "Turnover" in df.columns:
        mean_turnover = df["Turnover"].mean()
        std_turnover = df["Turnover"].std()
        df["IsAbnormalTurnover"] = (df["Turnover"] > mean_turnover + 3 * std_turnover).astype(int)
    else:
        df["IsAbnormalTurnover"] = 0
        print("⚠️ Turnover column not found, skipping IsAbnormalTurnover feature.")

    # Add future target variables for regression
    df["Future_Close"] = df["Close"].shift(-backtest_days)
    if "MainNetInflow" in df.columns:
        df["Future_MainNetInflow"] = df["MainNetInflow"].shift(-backtest_days)
    else:
        df["Future_MainNetInflow"] = np.nan

    # Add classification target
    df["Future_UpDown"] = (df["Future_Close"] > df["Close"]).astype(int)

    # Check for and handle NaNs in target variables before dropping NaNs
    if df["Future_Close"].isnull().any() or df["Future_MainNetInflow"].isnull().any() or df["Future_UpDown"].isnull().any():
        print(f"⚠️ 目标变量中存在 NaN 值，将在丢弃 NaN 后处理。")
        # Potentially add more sophisticated NaN handling if needed

    df.dropna(inplace=True)


    # 对目标变量进行归一化处理
    target_scaler_close = MinMaxScaler()
    target_scaler_fund = MinMaxScaler()

    y_close_scaled = target_scaler_close.fit_transform(df[["Future_Close"]])
    y_fund_scaled = target_scaler_fund.fit_transform(df[["Future_MainNetInflow"]])


    features = [
        "Open", "High", "Low", "Close", "Volume", "Amount", "Turnover",
        "MA5", "MA10", "EMA12", "EMA26", "MACD", "RSI",
        "Return_1d", "Volatility_5d", "Weekday", "IsMonthEnd",
        "MainNetInflow", "MainInflowRatio", "ChangePct",
        "MainNetInflow_Smooth", "MainNetInflow_Normalized",
        "MainNetInflow_Ratio_Change", "MainNetInflow_5d_Std",
        "IsLimitUp", "IsLimitDown", "ConsecutiveLimitUp_5d",
        "IsAbnormalInflow", "IsAbnormalTurnover"
    ]

    scaler = MinMaxScaler()
    df_scaled = scaler.fit_transform(df[features])
    return df_scaled, df, scaler, y_close_scaled.flatten(), y_fund_scaled.flatten(), df["Future_UpDown"].values, target_scaler_close, target_scaler_fund


# ✅ 1. 模型定义：Transformer 多输出架构


class MultiOutputTransformer(nn.Module):
    def __init__(self, input_dim, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.head_close = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1))  # 收盘价预测
        self.head_fund = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1))   # 资金流预测
        self.head_cls = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 2))    # 涨跌分类

    def forward(self, x):
        x_embed = self.embedding(x)
        x_encoded = self.encoder(x_embed)
        x_last = x_encoded[:, -1, :]
        return self.head_close(x_last), self.head_fund(x_last), self.head_cls(x_last)

# ✅ 2. 构建序列数据
def create_sequences(data, y_close, y_fund, y_cls, window_size):
    X, yc, yf, ycl = [], [], [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        yc.append(y_close[i+window_size])
        yf.append(y_fund[i+window_size])
        ycl.append(y_cls[i+window_size])
    return torch.tensor(X, dtype=torch.float32), torch.tensor(yc, dtype=torch.float32).unsqueeze(1), torch.tensor(yf, dtype=torch.float32).unsqueeze(1), torch.tensor(ycl, dtype=torch.long)

# ✅ 3. 训练函数 (Updated with validation, early stopping, and model saving)
def train_model_with_validation(model, train_loader, val_loader, epochs=150, lr=1e-3, patience=10, save_path="best_model.pt"):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)  # L2正则
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5) # Removed verbose
    loss_fn_reg = nn.MSELoss()
    loss_fn_cls = nn.CrossEntropyLoss()

    best_val_loss = float("inf")
    no_improve_epochs = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for X, yc, yf, ycl in train_loader:
            pred_c, pred_f, pred_cls = model(X)
            loss = loss_fn_reg(pred_c, yc) + loss_fn_reg(pred_f, yf) + loss_fn_cls(pred_cls, ycl)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # 验证阶段
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X, yc, yf, ycl in val_loader:
                pred_c, pred_f, pred_cls = model(X)
                loss = loss_fn_reg(pred_c, yc) + loss_fn_reg(pred_f, yf) + loss_fn_cls(pred_cls, ycl)
                val_loss += loss.item()

        scheduler.step(val_loss)
        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        # 保存最佳模型
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"📌 保存最佳模型（Val Loss: {best_val_loss:.4f}）到 {save_path}")
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            if no_improve_epochs >= patience:
                print(f"⏹️ 早停触发：验证集 {patience} 轮无提升")
                break


# ✅ 4. 未来预测函数（滚动方式）
def predict_future(model, last_sequence, pred_days=10):
    model.eval()
    preds_close, preds_fund, preds_cls = [], [], []
    seq = last_sequence.clone()

    for _ in range(pred_days):
        with torch.no_grad():
            pc, pf, pcl = model(seq.unsqueeze(0))
        preds_close.append(pc.item())
        preds_fund.append(pf.item())
        preds_cls.append(torch.argmax(pcl).item())

        # 滚动窗口更新 (Important: update with scaled values if needed for next prediction, or raw values and re-scale)
        # For simplicity here, we assume the model predicts scaled values.
        # If using predicted values as future input features, they would need to be scaled appropriately.
        # For this example, we'll just append a placeholder or the last actual data point if not predicting future inputs.
        # If you need to use predicted values as future inputs, you'll need a more complex loop
        # that appends the scaled predictions to the sequence for the next step.
        # For now, we will just append the last known sequence item for structural consistency,
        # although this is not a true autoregressive prediction using the model's output as input.
        next_step = seq[-1].clone() # Keep using the last actual data point for the sequence
        seq = torch.cat([seq[1:], next_step.unsqueeze(0)], dim=0)


    return preds_close, preds_fund, preds_cls

# ✅ 5. 可视化与保存

def plot_and_save_predictions(dates, preds_close_scaled, preds_fund_scaled, preds_cls, code, chart_dir, target_scaler_close, target_scaler_fund):
    # 反归一化预测值
    preds_close_real = target_scaler_close.inverse_transform(np.array(preds_close_scaled).reshape(-1, 1)).flatten()
    preds_fund_real = target_scaler_fund.inverse_transform(np.array(preds_fund_scaled).reshape(-1, 1)).flatten()

    df = pd.DataFrame({
        "Date": dates,
        "Predicted_Close": preds_close_real,
        "Predicted_FundFlow": preds_fund_real,
        "Predicted_UpDown": preds_cls
    })

    # 收盘价折线图
    plt.figure(figsize=(10, 4))
    plt.plot(df["Date"], df["Predicted_Close"], marker='o', color='red')
    plt.title(f"{code} 未来收盘价预测")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f"{chart_dir}/{code}_close_prediction.png")
    plt.close()

    # 资金流柱状图
    plt.figure(figsize=(10, 4))
    plt.bar(df["Date"], df["Predicted_FundFlow"], color='blue')
    plt.title(f"{code} 未来资金流预测")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f"{chart_dir}/{code}_fund_prediction.png")
    plt.close()

    # 分类标签保存
    df.to_csv(f"{chart_dir}/{code}_multi_predictions.csv", index=False)
    print(f"✅ 预测结果已保存：{chart_dir}/{code}_multi_predictions.csv")

# ✅ 6. 打包图表与结果
def zip_outputs(chart_dir, zip_name="predictions.zip"):
    with zipfile.ZipFile(zipf, 'w') as zipf:
        for fname in os.listdir(chart_dir):
            fpath = os.path.join(chart_dir, fname)
            zipf.write(fpath, arcname=fname)
    print(f"📦 所有图表与结果已打包：{zip_name}")


# Main execution loop
for code in stock_codes:
    print(f"Processing stock code: {code}")
    df_scaled, df_raw, scaler, y_close_scaled, y_fund_scaled, y_cls, target_scaler_close, target_scaler_fund = load_and_prepare_data(code, start_date, end_date)

    if df_scaled is None:
        continue  # Skip if data is missing

    X, yc, yf, ycl = create_sequences(df_scaled, y_close_scaled, y_fund_scaled, y_cls, window_size)

    # Split data into training and validation sets
    X_train, X_val, yc_train, yc_val, yf_train, yf_val, ycl_train, ycl_val = train_test_split(
        X, yc, yf, ycl, test_size=0.2, random_state=42, stratify=ycl # Stratify by classification target
    )

    train_dataset = TensorDataset(X_train, yc_train, yf_train, ycl_train)
    val_dataset = TensorDataset(X_val, yc_val, yf_val, ycl_val)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False) # No need to shuffle validation data

    model = MultiOutputTransformer(input_dim=X.shape[2])
    model_save_path = os.path.join(save_dir, f"{code}_best_model.pt")
    train_model_with_validation(model, train_loader, val_loader, save_path=model_save_path)

    # Load the best model before prediction
    model.load_state_dict(torch.load(model_save_path))

    # Predict future 10 days
    last_seq = X[-1]
    dates_future = pd.date_range(df_raw.index[-1] + pd.Timedelta(days=1), periods=backtest_days)
    preds_close_scaled, preds_fund_scaled, preds_cls = predict_future(model, last_seq, pred_days=backtest_days)

    # Visualize and save (pass scalers for inverse transform)
    plot_and_save_predictions(dates_future, preds_close_scaled, preds_fund_scaled, preds_cls, code, chart_dir, target_scaler_close, target_scaler_fund)

# Zip outputs after processing all codes
zip_outputs(chart_dir)

Processing stock code: 601138
⚠️ 目标变量中存在 NaN 值，将在丢弃 NaN 后处理。


/tmp/ipython-input-3142425109.py:95: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["MainNetInflow_Ratio_Change"] = df["MainInflowRatio"].pct_change().fillna(0)


Epoch 1 | Train Loss: 16.8770 | Val Loss: 4.2307
📌 保存最佳模型（Val Loss: 4.2307）到 ./models/601138_best_model.pt
Epoch 2 | Train Loss: 16.2921 | Val Loss: 4.2254
📌 保存最佳模型（Val Loss: 4.2254）到 ./models/601138_best_model.pt
Epoch 3 | Train Loss: 15.9566 | Val Loss: 4.2265
Epoch 4 | Train Loss: 15.7669 | Val Loss: 4.5690
Epoch 5 | Train Loss: 15.8619 | Val Loss: 4.1893
📌 保存最佳模型（Val Loss: 4.1893）到 ./models/601138_best_model.pt
Epoch 6 | Train Loss: 15.6801 | Val Loss: 4.1792
📌 保存最佳模型（Val Loss: 4.1792）到 ./models/601138_best_model.pt
Epoch 7 | Train Loss: 15.3893 | Val Loss: 4.1740
📌 保存最佳模型（Val Loss: 4.1740）到 ./models/601138_best_model.pt
Epoch 8 | Train Loss: 15.4569 | Val Loss: 4.1493
📌 保存最佳模型（Val Loss: 4.1493）到 ./models/601138_best_model.pt
Epoch 9 | Train Loss: 15.9313 | Val Loss: 4.1690
Epoch 10 | Train Loss: 15.0600 | Val Loss: 4.0371
📌 保存最佳模型（Val Loss: 4.0371）到 ./models/601138_best_model.pt
Epoch 11 | Train Loss: 15.1662 | Val Loss: 4.2597
Epoch 12 | Train Loss: 15.3394 | Val Loss: 3.9725
📌 保

KeyboardInterrupt: 

In [11]:
import pandas as pd

df_predictions = pd.read_csv('/content/charts/603725_multi_predictions.csv')
display(df_predictions.head())

,Date,Predicted_Close,Predicted_FundFlow,Predicted_UpDown
0,2025-09-26,9.607475,4.953490,0
1,2025-09-27,9.609224,4.926586,0
2,2025-09-28,9.612956,4.888198,0
3,2025-09-29,9.614235,4.851395,0
4,2025-09-30,9.616412,4.810615,0
